### **Notebook 3 (etapa 5): SHAP estratificado para cada categoría de cáncer (con muestra mayor a 100 pacientes)**

In [ ]:
import os  # Interacción con el sistema operativo (creación de directorios y manejo de rutas)
import gc  # Recolección de basura (Garbage Collector) para liberar memoria RAM durante el bucle
import time  # Medición de tiempos de ejecución para monitorear el progreso
import pickle  # Serialización nativa de Python (importado por compatibilidad de dependencias)
import joblib  # Carga (deserialización) del modelo de Machine Learning pre-entrenado
import numpy as np  # Facilita la realización de cálculos numéricos avanzados y manejo de matrices
import pandas as pd  # Permite el manejo y análisis de estructuras de datos tabulares (DataFrames)
import shap  # Biblioteca principal basada en teoría de juegos para la explicabilidad de modelos
import matplotlib.pyplot as plt  # Biblioteca para la creación y exportación de visualizaciones gráficas
import warnings  # Control de advertencias del sistema
warnings.filterwarnings("ignore", category=UserWarning)  # Suprime advertencias no críticas en la consola

def generar_shap_estratificado_cancer_rf():
    """
    Descripción:
        Ejecuta un pipeline SHAP para generar análisis predictivos y de explicabilidad específicos 
        por cada TIPO DE CÁNCER de forma aislada, utilizando el modelo Random Forest (Mortalidad).
        Maneja dinámicamente la categoría base (C00_C14), filtra grupos pequeños mediante un 
        umbral de robustez estadística (mínimo 100 pacientes), y genera salidas absolutas, 
        porcentuales, direccionales y gráficos de dependencia por cada subgrupo.

    Entradas:
        - Ninguna explícita: La función consume directamente el dataset oncológico de prueba 
          y el modelo .pkl desde el disco duro.

    Salidas:
        - None: La función no retorna variables en memoria, pero genera una estructura de carpetas
          (una por cada tipo de cáncer viable) conteniendo:
            1. Matrices SHAP crudas (.npy).
            2. Reportes CSV de impacto absoluto, porcentual y direccional (numérico y categórico).
            3. Gráficos PNG de resumen (Top 10) y paneles de dependencia específicos del subgrupo.
    """
    # Definir la variable objetivo fija para este análisis
    target_name = 'MORTALIDAD'
    
    # -------------------------------------------------------------------------
    # CONFIGURACIÓN DE RUTAS
    # -------------------------------------------------------------------------
    # Directorios de origen de los datos y modelos
    dir_datos = "../../Datos/Datasets Finales"
    dir_modelos = "../../Resultados/Resultados (etapa 3 y 4)/Random_Forest"
    
    # Directorio destino principal para los resultados estratificados
    dir_base_estratificado = f"../../Resultados/Resultados (etapa 5)/SHAP_{target_name}/Estratificado_Por_Cancer"
    os.makedirs(dir_base_estratificado, exist_ok=True)
    
    # Construir ruta exacta hacia el modelo Random Forest
    nombre_modelo = f"Modelo_Optimo_RF_{target_name}.pkl"
    ruta_modelo = os.path.join(dir_modelos, nombre_modelo)
    
    # Listas de variables a excluir del análisis y variables numéricas continuas
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER']
    vars_num = ['CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS']
    
    # Lista exhaustiva de las categorías (One-Hot) de cáncer a iterar
    categorias_cancer = [
        'CATEGORIA_CANCER_C00_C14', 'CATEGORIA_CANCER_C15_C26', 'CATEGORIA_CANCER_C30_C39', 
        'CATEGORIA_CANCER_C40_C41', 'CATEGORIA_CANCER_C43_C44', 'CATEGORIA_CANCER_C45_C49', 
        'CATEGORIA_CANCER_C50', 'CATEGORIA_CANCER_C51_C58', 'CATEGORIA_CANCER_C60_C63', 
        'CATEGORIA_CANCER_C64_C68', 'CATEGORIA_CANCER_C69_C72', 'CATEGORIA_CANCER_C73_C75', 
        'CATEGORIA_CANCER_C76_C80', 'CATEGORIA_CANCER_C81_C96', 'CATEGORIA_CANCER_C97'
    ]

    # Imprimir encabezado de la ejecución
    print("="*80)
    print(f"INICIANDO SHAP ESTRATIFICADO POR TIPO DE CÁNCER (TOP 10) - TARGET: {target_name}")
    print(f"Hora de inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    # Validar que el archivo del modelo exista antes de iniciar el procesamiento masivo
    if not os.path.exists(ruta_modelo):
        print(f"ERROR: No se encontró el modelo en la ruta: {ruta_modelo}")
        return
        
    print(f"-> Cargando modelo óptimo pre-entrenado...")
    # Deserializar y cargar el modelo en RAM
    modelo_rf = joblib.load(ruta_modelo)
        
    # Desempaquetador dinámico: Extraer el modelo real si está envuelto en Pipelines, GridSearchCV o Calibradores
    if isinstance(modelo_rf, dict):
        for k, v in modelo_rf.items():
            if hasattr(v, 'predict'): modelo_rf = v; break
    if isinstance(modelo_rf, (list, tuple)):
        for v in modelo_rf:
            if hasattr(v, 'predict'): modelo_rf = v; break
    if hasattr(modelo_rf, 'best_estimator_'): modelo_rf = modelo_rf.best_estimator_
    if hasattr(modelo_rf, 'steps'): modelo_rf = modelo_rf.steps[-1][1]
    if hasattr(modelo_rf, 'calibrated_classifiers_'): modelo_rf = modelo_rf.calibrated_classifiers_[0].estimator

    # Recuperar el orden estricto de las características desde el modelo o infiriéndolo del dataset
    if hasattr(modelo_rf, 'feature_names_in_'):
        features = modelo_rf.feature_names_in_.tolist()
    else:
        df_dummy = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), nrows=1)
        features = [c for c in df_dummy.columns if c not in cols_excluir]

    print("-> Inicializando SHAP TreeExplainer nativo...")
    # Instanciar el explicador de árboles SHAP
    explainer = shap.TreeExplainer(modelo_rf)

    print("-> Cargando el 100% de la Cohorte Oncológica de Evaluación...")
    # Cargar los datos de prueba
    df_onco_completo = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)

    # Identificar las columnas One-Hot de cáncer que realmente se materializaron en el dataset
    cols_cancer_reales = [col for col in df_onco_completo.columns if col.startswith('CATEGORIA_CANCER_')]

    # -------------------------------------------------------------------------
    # BUCLE PRINCIPAL POR CATEGORÍA DE CÁNCER
    # -------------------------------------------------------------------------
    for categoria in categorias_cancer:
        
        # LÓGICA ESPECIAL PARA LA CATEGORÍA BASE (C00_C14)
        # Como es la categoría "drop_first", se asume cuando todas las demás OHE de cáncer son 0
        if categoria == 'CATEGORIA_CANCER_C00_C14':
            mascara_base = df_onco_completo[cols_cancer_reales].sum(axis=1) == 0
            df_filtrado = df_onco_completo[mascara_base].copy()
            
        # LÓGICA NORMAL PARA LAS DEMÁS CATEGORÍAS (Se filtran donde la columna OHE == 1)
        else:
            if categoria not in df_onco_completo.columns:
                print(f"\nAVISO: La columna {categoria} no existe. Saltando...")
                continue
            df_filtrado = df_onco_completo[df_onco_completo[categoria] == 1].copy()
            
        # Determinar el tamaño muestral del subgrupo actual
        n_pacientes = len(df_filtrado)
        # Limpiar el string para usarlo en los nombres de archivos y carpetas
        nombre_limpio = categoria.replace('CATEGORIA_CANCER_', '')
        # Umbral mínimo de pacientes para considerar que el análisis SHAP tiene robustez estadística
        UMBRAL_MINIMO = 100
        
        # Filtro de robustez: Omitir subgrupos pequeños para evitar conclusiones espurias
        if n_pacientes < UMBRAL_MINIMO:
            print(f"\nAVISO: La categoría {nombre_limpio} no supera el umbral mínimo de {UMBRAL_MINIMO} personas. Excluyendo por falta de robustez estadística.")
            continue
        
        # Anunciar inicio del análisis para el subgrupo aprobado
        print("\n" + "-"*60)
        print(f"--- PROCESANDO SUBGRUPO: {nombre_limpio} ({n_pacientes} pacientes) ---")
        print("-"*60)
            
        # Configurar carpetas específicas para el subgrupo
        dir_sub_enfoque = os.path.join(dir_base_estratificado, nombre_limpio)
        dir_dependence = os.path.join(dir_sub_enfoque, "Dependence_Plots")
        os.makedirs(dir_dependence, exist_ok=True)
        
        # Alinear matriz predictora y limpiar variables temporales
        X_shap = df_filtrado[features].astype('float32')
        del df_filtrado; gc.collect()
        
        # Iniciar cronómetro de SHAP
        inicio_time = time.time()
        
        # Calcular SHAP en bloques pequeños (500) para proteger la RAM con Random Forest
        batch_size = 500
        resultados_list = []
        n_batches = (len(X_shap) // batch_size) + (1 if len(X_shap) % batch_size != 0 else 0)
        
        # Bucle de cálculo por lotes
        for i in range(0, len(X_shap), batch_size):
            batch = X_shap.iloc[i:i+batch_size]
            if (i // batch_size + 1) % 10 == 0 or (i // batch_size + 1) == 1:
                print(f"      -> Bloque {i//batch_size + 1} de {n_batches}...")
            
            # Aproximar SHAP para acelerar el cómputo conservando la tendencia general
            shap_output = explainer.shap_values(batch, check_additivity=False, approximate=True)
            
            # Formatear la salida SHAP a un tensor tridimensional homogéneo (Muestras, Variables, Clases)
            if isinstance(shap_output, list):
                shap_mat_batch = np.stack(shap_output, axis=2)
            elif len(shap_output.shape) == 3:
                shap_mat_batch = shap_output
            else:
                shap_mat_batch = np.stack([shap_output * -1, shap_output], axis=2)
                
            resultados_list.append(shap_mat_batch)
            del batch, shap_output, shap_mat_batch; gc.collect()
            
        # Concatenar todos los bloques
        matriz_shap = np.concatenate(resultados_list, axis=0)
        print(f"   -> SHAP calculado en {round((time.time() - inicio_time)/60, 2)} minutos.")
        
        # --- FILTRO AUTOMÁTICO DE CONSTANTES ---
        # Si un tipo de cáncer es exclusivo de un sexo (ej. Cáncer de Mama / C50), la variable sexo no tendrá varianza.
        # Este filtro remueve características sin varianza dentro del subgrupo.
        varianzas = X_shap.var()
        cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
        
        if cols_a_eliminar:
            idx_a_eliminar = [X_shap.columns.get_loc(col) for col in cols_a_eliminar]
            X_shap = X_shap.drop(columns=cols_a_eliminar)
            matriz_shap = np.delete(matriz_shap, idx_a_eliminar, axis=1)
            print(f"   -> Se excluyeron {len(cols_a_eliminar)} variables constantes para este subgrupo.")
        
        # Identificar número de clases de la matriz
        n_clases = matriz_shap.shape[2]
        
        # Guardar respaldo numérico del subgrupo
        ruta_npy = os.path.join(dir_sub_enfoque, f"MATRIZ_SHAP_{nombre_limpio}.npy")
        np.save(ruta_npy, matriz_shap)
        
        # Exportar CSV de Importancias (Absoluto)
        shap_abs = np.abs(matriz_shap).mean(axis=0)
        impacto_total = shap_abs.sum(axis=1)
        
        # Etiquetar columnas dinámicamente según la clase (0: Vivo, 1: Fallecido)
        columnas_csv = [f"Clase_{i}_" + ("Vivo" if i==0 else "Fallecido") for i in range(n_clases)]
        df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=columnas_csv)
        df_shap_imp['Impacto_Total'] = impacto_total
        # Ordenar el dataframe por impacto
        df_shap_imp = df_shap_imp.sort_values(by='Impacto_Total', ascending=False)
        
        # Exportar ranking a CSV
        ruta_csv = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{nombre_limpio}.csv")
        df_shap_imp.to_csv(ruta_csv, index_label='Variable')
        
        # CSV en Porcentajes relativos
        print("   -> Generando CSV de importancias en porcentajes...")
        df_shap_porcentajes = (df_shap_imp / df_shap_imp.sum()) * 100
        ruta_csv_pct = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{nombre_limpio}_PORCENTAJES.csv")
        df_shap_porcentajes.to_csv(ruta_csv_pct, index_label='Variable')
        
        # Gráfico Summary General (Top 10) específico para el subgrupo
        plt.figure(figsize=(10, 6))
        df_top10 = df_shap_imp.head(10).drop(columns=['Impacto_Total']).iloc[::-1]
        df_top10.plot(kind='barh', stacked=True, figsize=(10, 6), cmap='bwr', ax=plt.gca()) 
        plt.title(f'Top 10 Variables SHAP - Cáncer {nombre_limpio} (Mortalidad)', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_General_{nombre_limpio}.png"), dpi=300)
        plt.close()
        
        # Rangos Numéricos Absolutos
        # Calcula cuánto SHAP aporta cada variable continua fragmentándola en 4 cuartiles
        print("   -> Calculando impacto absoluto por rangos numéricos...")
        rangos_resumen = []
        for v_num in vars_num:
            if v_num in X_shap.columns:
                try:
                    bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except:
                    bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                # Iterar sobre cada cuartil
                for idx_c, rango in enumerate(bins_serie.cat.categories):
                    indices_rango = (bins_serie == rango)
                    if indices_rango.sum() > 0:
                        # Extraer media del impacto absoluto para este rango específico
                        impacto_medio_rango = np.abs(matriz_shap[indices_rango, X_shap.columns.get_loc(v_num), :]).mean(axis=0)
                        dict_rango = {"Variable": v_num, "Rango": str(rango), "N_Pacientes": indices_rango.sum()}
                        # Agregar las medias asociadas a cada clase
                        for cl in range(n_clases):
                            etiqueta = "Vivo" if cl == 0 else "Fallecido"
                            dict_rango[f"Impacto_Promedio_Clase_{cl}_{etiqueta}"] = impacto_medio_rango[cl]
                        rangos_resumen.append(dict_rango)
                            
        # Exportar resumen absoluto de variables continuas
        df_rangos = pd.DataFrame(rangos_resumen)
        df_rangos.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Impacto_Variables_Numericas_{nombre_limpio}.csv"), index=False)
        
        # -------------------------------------------------------------------------
        # ANÁLISIS DIRECCIONAL: VARIABLES NUMÉRICAS
        # -------------------------------------------------------------------------
        print("   -> Calculando impacto direccional (Mortalidad) por rangos numéricos...")
        rangos_direccionales = []
        # Aislar tensor enfocado únicamente en la clase 1 (Fallecido)
        matriz_fallecido = matriz_shap[:, :, 1]
        
        for v_num in vars_num:
            if v_num in X_shap.columns:
                try:
                    bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except:
                    bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                idx_var = X_shap.columns.get_loc(v_num)
                
                for rango in bins_serie.cat.categories:
                    indices_rango = (bins_serie == rango)
                    n_pacientes_rango = indices_rango.sum()
                    
                    if n_pacientes_rango > 0:
                        # Evaluar si el promedio en este cuartil aumenta o disminuye el riesgo (+ o -)
                        valores_crudos = matriz_fallecido[indices_rango, idx_var]
                        promedio_crudo = valores_crudos.mean()
                        
                        if promedio_crudo > 0:
                            efecto = "Aumenta Mortalidad (+)"
                        elif promedio_crudo < 0:
                            efecto = "Protector / Supervivencia (-)"
                        else:
                            efecto = "Neutral"

                        # Acumular resultados direccionales
                        rangos_direccionales.append({
                            "Variable": v_num,
                            "Rango": str(rango),
                            "N_Pacientes": n_pacientes_rango,
                            "SHAP_Promedio_Crudo_Mortalidad": promedio_crudo,
                            "Efecto_Clinico": efecto
                        })
                        
        # Exportar CSV de direccionalidad numérica
        df_direccional = pd.DataFrame(rangos_direccionales)
        df_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccionales_Mortalidad_{nombre_limpio}.csv"), index=False)

        # -------------------------------------------------------------------------
        # ANÁLISIS DIRECCIONAL: VARIABLES CATEGÓRICAS (OHE)
        # -------------------------------------------------------------------------
        print("   -> Calculando impacto direccional (Mortalidad) para variables categóricas (OHE)...")
        vars_cat_ohe = [col for col in X_shap.columns if col not in vars_num]
        cat_direccionales = []
        
        for v_cat in vars_cat_ohe:
            idx_var = X_shap.columns.get_loc(v_cat)
            # Evaluar impacto en pacientes SIN la condición (0) vs CON la condición (1)
            for valor_cat in [0, 1]:
                indices_cat = (X_shap[v_cat] == valor_cat)
                n_pacientes_cat = indices_cat.sum()
                
                if n_pacientes_cat > 0:
                    promedio_crudo = matriz_fallecido[indices_cat, idx_var].mean()
                    
                    if promedio_crudo > 0:
                        efecto = "Aumenta Mortalidad (+)"
                    elif promedio_crudo < 0:
                        efecto = "Protector / Supervivencia (-)"
                    else:
                        efecto = "Neutral"

                    cat_direccionales.append({
                        "Variable": v_cat,
                        "Condicion_OHE": valor_cat,
                        "Significado": "Presencia (1)" if valor_cat == 1 else "Ausencia (0)",
                        "N_Pacientes": n_pacientes_cat,
                        "SHAP_Promedio_Crudo_Mortalidad": promedio_crudo,
                        "Efecto_Clinico": efecto
                    })
                    
        # Exportar CSV de direccionalidad categórica
        df_cat_direccional = pd.DataFrame(cat_direccionales)
        df_cat_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Categoricas_Mortalidad_{nombre_limpio}.csv"), index=False)

        # -------------------------------------------------------------------------
        # PANELES DE DEPENDENCIA (TOP 10)
        # -------------------------------------------------------------------------
        print(f"   -> Generando paneles de dependencia para el Top 10...")
        # Tomar los 10 predictores más influyentes de este subgrupo de cáncer
        top_10_vars = df_shap_imp.head(10).index.tolist()
        
        for var in top_10_vars:
            if var in X_shap.columns:
                fig, ax = plt.subplots(figsize=(6, 4.5))
                valores_sh_fallecido = matriz_shap[:, :, 1]
                
                # Crear gráfico de dispersión cruzando el valor de la característica vs su impacto SHAP
                shap.dependence_plot(
                    var, valores_sh_fallecido, X_shap, 
                    interaction_index=None, ax=ax, show=False
                )
                ax.set_title(f'Impacto en Riesgo de Fallecimiento (Clase 1)', fontsize=10)
                fig.suptitle(f'Dependence Plot: {var} ({nombre_limpio})', fontsize=11, y=1.02)
                plt.tight_layout()
                plt.savefig(os.path.join(dir_dependence, f"SHAP_Dependence_{var}.png"), dpi=200, bbox_inches='tight')
                plt.close()
                
        # Limpiar variables generadas masivamente en esta iteración (subgrupo) para evitar colapsos
        del X_shap, matriz_shap, df_shap_imp, df_shap_porcentajes, df_direccional, df_cat_direccional; gc.collect()

    # Anunciar el fin de todo el proceso iterativo
    print("\n" + "="*80)
    print("PROCESO DE ESTRATIFICACIÓN CONCLUIDO CON ÉXITO")
    print(f"Resultados guardados en:\n{dir_base_estratificado}")
    print("="*80)
    
    # Liberar el dataset inicial completo de la RAM
    del df_onco_completo; gc.collect()

# Ejecutar el script estratificado
generar_shap_estratificado_cancer_rf()

INICIANDO SHAP ESTRATIFICADO POR TIPO DE CÁNCER (TOP 10) - TARGET: MORTALIDAD
Hora de inicio: 2026-07-16 06:25:33
-> Cargando modelo óptimo pre-entrenado...
-> Inicializando SHAP TreeExplainer nativo...
-> Cargando el 100% de la Cohorte Oncológica de Evaluación...

AVISO: La categoría C00_C14 no supera el umbral mínimo de 100 personas. Excluyendo por falta de robustez estadística.

------------------------------------------------------------
--- PROCESANDO SUBGRUPO: C15_C26 (26320 pacientes) ---
------------------------------------------------------------
      -> Bloque 1 de 53...
      -> Bloque 10 de 53...
      -> Bloque 20 de 53...
      -> Bloque 30 de 53...
      -> Bloque 40 de 53...
      -> Bloque 50 de 53...
   -> SHAP calculado en 0.48 minutos.
   -> Se excluyeron 17 variables constantes.
   -> Generando CSV de importancias en porcentajes...
   -> Calculando impacto absoluto por rangos numéricos...
   -> Calculando impacto direccional (Mortalidad) por rangos numéricos...
  

In [ ]:
import os  # Interacción con el sistema operativo (creación de directorios y manejo de rutas)
import gc  # Recolección de basura (Garbage Collector) para liberar memoria RAM durante el procesamiento por lotes
import time  # Medición de tiempos de ejecución para monitorear el progreso del pipeline
import pickle  # Serialización nativa de Python para cargar el modelo XGBoost pre-entrenado
import numpy as np  # Facilita cálculos numéricos avanzados y el manejo de matrices multidimensionales (tensores)
import pandas as pd  # Permite el manejo, filtrado y análisis de estructuras de datos tabulares (DataFrames)
import shap  # Biblioteca principal basada en teoría de juegos para la explicabilidad de los modelos
import matplotlib.pyplot as plt  # Biblioteca para la creación, formateo y exportación de visualizaciones gráficas
import warnings  # Control de advertencias del sistema
warnings.filterwarnings("ignore", category=UserWarning)  # Suprime advertencias no críticas para mantener limpia la consola

def generar_shap_estratificado_cancer_xgb(target_name):
    """
    Descripción:
        Ejecuta un pipeline SHAP para generar análisis predictivos y de explicabilidad estratificados 
        por cada TIPO DE CÁNCER de forma aislada, utilizando modelos XGBoost Multiclase.
        Ajusta el foco clínico dinámicamente según el target (Severidad o Consumo), filtra 
        subgrupos pequeños mediante un umbral de robustez estadística (<100 pacientes), maneja la 
        categoría base (C00_C14), y extrae rangos direccionales (numéricos y categóricos) 
        enfocados específicamente en la clase de mayor riesgo.

    Entradas:
        - target_name (str): Nombre de la variable objetivo multiclase a evaluar ('SEVERIDAD' o 'CONSUMO_RECURSOS').

    Salidas:
        - None: La función no retorna variables en memoria, pero genera una estructura de subcarpetas 
          (una por cada tipo de cáncer que supere el umbral) conteniendo:
            1. Matrices SHAP crudas multidimensionales (.npy).
            2. Reportes CSV de impacto absoluto, porcentual y direccional (numérico y categórico).
            3. Gráficos PNG de resumen (Top 10) y paneles de dependencia específicos del subgrupo.
    """
    # -------------------------------------------------------------------------
    # CONFIGURACIÓN DINÁMICA POR TARGET (CLASE CRÍTICA)
    # -------------------------------------------------------------------------
    # Configuración dinámica del índice de mayor riesgo según el target analizado
    if target_name == 'SEVERIDAD':
        idx_clase_alta = 3  # Clase 3 corresponde a Severidad Mayor (índices: 0, 1, 2, 3)
        nombre_efecto_str = 'Severidad Alta (Clase 3)'
    else:
        idx_clase_alta = 2  # Clase 2 corresponde a Consumo Alto (índices: 0, 1, 2)
        nombre_efecto_str = 'Consumo Alto (Clase 2)'

    # -------------------------------------------------------------------------
    # CONFIGURACIÓN DE RUTAS
    # -------------------------------------------------------------------------
    # Directorios de origen para los datos y modelos entrenados
    dir_datos = "../../Datos/Datasets Finales"
    dir_modelos = "../../Resultados/Resultados (etapa 3 y 4)/XGBoost"
    
    # Directorio destino principal para los resultados estratificados de esta etapa
    dir_base_estratificado = f"../../Resultados/Resultados (etapa 5)/SHAP_{target_name}/Estratificado_Por_Cancer"
    os.makedirs(dir_base_estratificado, exist_ok=True)
    
    # Construir ruta exacta hacia el modelo XGBoost óptimo
    nombre_modelo = f"Modelo_Optimo_XGBoost_{target_name}.pkl"
    ruta_modelo = os.path.join(dir_modelos, nombre_modelo)
    
    # Listas de variables a excluir del análisis y variables estrictamente numéricas (continuas)
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER']
    vars_num = ['CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS']
    
    # Lista exhaustiva de las categorías (One-Hot) de cáncer a iterar en el proceso
    categorias_cancer = [
        'CATEGORIA_CANCER_C00_C14', 'CATEGORIA_CANCER_C15_C26', 'CATEGORIA_CANCER_C30_C39',
        'CATEGORIA_CANCER_C40_C41', 'CATEGORIA_CANCER_C43_C44', 'CATEGORIA_CANCER_C45_C49',
        'CATEGORIA_CANCER_C50', 'CATEGORIA_CANCER_C51_C58', 'CATEGORIA_CANCER_C60_C63',
        'CATEGORIA_CANCER_C64_C68', 'CATEGORIA_CANCER_C69_C72', 'CATEGORIA_CANCER_C73_C75',
        'CATEGORIA_CANCER_C76_C80', 'CATEGORIA_CANCER_C81_C96', 'CATEGORIA_CANCER_C97'
    ]

    # Imprimir encabezado de la ejecución en consola
    print("="*80)
    print(f"INICIANDO SHAP ESTRATIFICADO POR CÁNCER (TOP 10) - TARGET: {target_name} (XGBOOST)")
    print(f"Enfocando análisis direccional en: {nombre_efecto_str}")
    print(f"Hora de inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    # Validar que el modelo exista físicamente antes de iniciar la iteración masiva
    if not os.path.exists(ruta_modelo):
        print(f"ERROR: No se encontró el modelo óptimo en la ruta: {ruta_modelo}")
        return
        
    print(f"-> Cargando modelo XGBoost pre-entrenado desde: {nombre_modelo}...")
    # Deserializar y cargar el modelo XGBoost desde el archivo local
    with open(ruta_modelo, 'rb') as f:
        modelo_xgb = pickle.load(f)
        
    print("-> Inicializando SHAP TreeExplainer nativo...")
    # Instanciar el explicador óptimo de SHAP para modelos de árboles de decisión
    explainer = shap.TreeExplainer(modelo_xgb)
    # Extraer el orden estricto de las variables directamente del motor interno de XGBoost
    features = modelo_xgb.get_booster().feature_names

    print("-> Cargando el 100% de la Cohorte Oncológica de Evaluación...")
    # Cargar el dataset de prueba completo
    df_onco_completo = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)

    # Identificar dinámicamente las columnas One-Hot de cáncer que realmente existen en el dataset
    cols_cancer_reales = [col for col in df_onco_completo.columns if col.startswith('CATEGORIA_CANCER_')]

    # -------------------------------------------------------------------------
    # BUCLE PRINCIPAL POR CATEGORÍA DE CÁNCER
    # -------------------------------------------------------------------------
    for categoria in categorias_cancer:
        
        # LÓGICA ESPECIAL PARA LA CATEGORÍA BASE (C00_C14)
        # Identifica a los pacientes donde todas las demás categorías de cáncer son 0 (drop_first concept)
        if categoria == 'CATEGORIA_CANCER_C00_C14':
            mascara_base = df_onco_completo[cols_cancer_reales].sum(axis=1) == 0
            df_filtrado = df_onco_completo[mascara_base].copy()
            
        # LÓGICA NORMAL PARA LAS DEMÁS CATEGORÍAS
        else:
            if categoria not in df_onco_completo.columns:
                print(f"\nADVERTENCIA: La columna {categoria} no existe. Saltando...")
                continue
            # Filtrar solo los pacientes que presentan la categoría de cáncer actual
            df_filtrado = df_onco_completo[df_onco_completo[categoria] == 1].copy()
            
        # Determinar el tamaño muestral del subgrupo
        n_pacientes = len(df_filtrado)
        # Limpiar el prefijo para utilizar el nombre en la creación de archivos y carpetas
        nombre_limpio = categoria.replace('CATEGORIA_CANCER_', '')
        # Umbral estadístico mínimo de pacientes para justificar la explicabilidad local
        UMBRAL_MINIMO = 100
        
        # Omitir subgrupos pequeños para prevenir conclusiones clínicas basadas en ruido
        if n_pacientes < UMBRAL_MINIMO:
            print(f"\nADVERTENCIA: La categoría {nombre_limpio} no supera el umbral mínimo de {UMBRAL_MINIMO} personas. Excluyendo por falta de robustez estadística.")
            continue
            
        # Anunciar inicio del procesamiento del subgrupo válido
        print("\n" + "-"*60)
        print(f"--- PROCESANDO SUBGRUPO: {nombre_limpio} ({n_pacientes} pacientes) ---")
        print("-"*60)
        
        # Configurar carpetas de resultados específicas para este cáncer
        dir_sub_enfoque = os.path.join(dir_base_estratificado, nombre_limpio)
        dir_dependence = os.path.join(dir_sub_enfoque, "Dependence_Plots")
        os.makedirs(dir_dependence, exist_ok=True)
        
        # Alinear la matriz predictora al formato del modelo y limpiar RAM
        X_shap = df_filtrado[features].astype('float32')
        del df_filtrado; gc.collect()
        
        # Iniciar cronómetro de cómputo SHAP para este subgrupo
        inicio_time = time.time()
        
        # Procesar en bloques grandes (XGBoost gestiona mejor la RAM que Random Forest en SHAP)
        batch_size = 10000
        resultados_list = []
        n_batches = (len(X_shap) // batch_size) + (1 if len(X_shap) % batch_size != 0 else 0)
        
        # Bucle de cálculo por lotes
        for i in range(0, len(X_shap), batch_size):
            batch = X_shap.iloc[i:i+batch_size]
            print(f"      -> Procesando bloque {i//batch_size + 1} de {n_batches}...")
            
            # Calcular objeto SHAP completo
            shap_obj = explainer(batch)
            resultados_list.append(shap_obj.values)
            # Limpiar lote de la memoria
            del batch, shap_obj; gc.collect()
            
        # Unir todos los lotes procesados en una matriz tridimensional final
        matriz_shap = np.concatenate(resultados_list, axis=0)
        print(f"   -> SHAP completado en {round((time.time() - inicio_time)/60, 2)} minutos.")
        
        # --- FILTRO AUTOMÁTICO DE CONSTANTES ---
        # Evaluar y remover características sin varianza en el subgrupo (ej. tipos de cáncer específicos de sexo)
        varianzas = X_shap.var()
        cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
        
        # Seguridad: Forzar remoción del flag 'SIN_CANCER' si por anomalía del dataset siguiera presente
        if 'CATEGORIA_CANCER_SIN_CANCER' in X_shap.columns and 'CATEGORIA_CANCER_SIN_CANCER' not in cols_a_eliminar:
            cols_a_eliminar.append('CATEGORIA_CANCER_SIN_CANCER')
            
        # Purgar columnas constantes de los DataFrames y del tensor SHAP
        if cols_a_eliminar:
            idx_a_eliminar = [X_shap.columns.get_loc(col) for col in cols_a_eliminar]
            X_shap = X_shap.drop(columns=cols_a_eliminar)
            
            # Lógica dinámica según las dimensiones que devuelva SHAP (XGBoost Multiclase es 3D)
            if len(matriz_shap.shape) == 3: 
                matriz_shap = np.delete(matriz_shap, idx_a_eliminar, axis=1)
            else:
                matriz_shap = np.delete(matriz_shap, idx_a_eliminar, axis=1)
                
            print(f"   -> Se excluyeron {len(cols_a_eliminar)} variables constantes.")
        
        # Extraer número de clases dinámicamente de las dimensiones del tensor
        n_clases = matriz_shap.shape[2] if len(matriz_shap.shape) == 3 else 1
        
        # Guardar respaldo de la matriz cruda (.npy)
        ruta_npy = os.path.join(dir_sub_enfoque, f"MATRIZ_SHAP_{nombre_limpio}.npy")
        np.save(ruta_npy, matriz_shap)
        
        # Exportar CSV de Importancias Agregadas (Magnitud Absoluta)
        shap_abs = np.abs(matriz_shap).mean(axis=0) 
        if n_clases > 1:
            # En multiclase, crear una puntuación sumada de impacto global
            impacto_total = shap_abs.sum(axis=1)
            columnas_csv = [f"Clase_{i}" for i in range(n_clases)]
            df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=columnas_csv)
            df_shap_imp['Impacto_Total'] = impacto_total
        else:
            df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=['Impacto_Total'])
            
        # Ordenar el dataset por relevancia clínica y guardar
        df_shap_imp = df_shap_imp.sort_values(by='Impacto_Total', ascending=False)
        ruta_csv = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{nombre_limpio}.csv")
        df_shap_imp.to_csv(ruta_csv, index_label='Variable')
        
        # CSV en Porcentajes Relativos
        print("   -> Generando CSV de importancias en porcentajes...")
        df_shap_porcentajes = (df_shap_imp / df_shap_imp.sum()) * 100
        ruta_csv_pct = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{nombre_limpio}_PORCENTAJES.csv")
        df_shap_porcentajes.to_csv(ruta_csv_pct, index_label='Variable')
        
        # Gráfico Summary General Stacked (TOP 10)
        plt.figure(figsize=(10, 6))
        df_top10 = df_shap_imp.head(10)
        # Adaptar renderizado dependiendo de si es multiclase (barras apiladas) o binario
        if n_clases > 1:
            df_top10 = df_top10.drop(columns=['Impacto_Total']).iloc[::-1]
            df_top10.plot(kind='barh', stacked=True, figsize=(10, 6), cmap='viridis', ax=plt.gca())
        else:
            df_top10['Impacto_Total'].iloc[::-1].plot(kind='barh', figsize=(10, 6), color='teal', ax=plt.gca())
            
        plt.title(f'Top 10 Variables SHAP - Cáncer {nombre_limpio} ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_General_{nombre_limpio}.png"), dpi=300)
        plt.close()
        
        # Gráfico Summary Plot Filtrado Solo a Variables Categóricas (TOP 10)
        plt.figure(figsize=(10, 6))
        vars_cat_ohe = [col for col in df_shap_imp.index if col not in vars_num]
        df_top10_cat = df_shap_imp.loc[vars_cat_ohe].head(10)
        if n_clases > 1:
            df_top10_cat = df_top10_cat.drop(columns=['Impacto_Total']).iloc[::-1]
            df_top10_cat.plot(kind='barh', stacked=True, figsize=(10, 6), cmap='plasma', ax=plt.gca())
        else:
            df_top10_cat['Impacto_Total'].iloc[::-1].plot(kind='barh', figsize=(10, 6), color='purple', ax=plt.gca())
            
        plt.title(f'Top 10 Categóricas SHAP - Cáncer {nombre_limpio} ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_Categoricas_{nombre_limpio}.png"), dpi=300)
        plt.close()
        
        # Análisis cuantitativo absoluto por rangos
        print("   -> Calculando impacto absoluto por rangos numéricos...")
        rangos_resumen = []
        for v_num in vars_num:
            if v_num in X_shap.columns:
                # Segmentar distribución en 4 cuartiles
                try:
                    bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except:
                    bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                # Desglosar promedios absolutos por clase dentro de cada cuartil
                for idx_c, rango in enumerate(bins_serie.cat.categories):
                    indices_rango = (bins_serie == rango)
                    if indices_rango.sum() > 0:
                        if n_clases > 1:
                            impacto_medio_rango = np.abs(matriz_shap[indices_rango, X_shap.columns.get_loc(v_num), :]).mean(axis=0)
                            dict_rango = {"Variable": v_num, "Rango": str(rango), "N_Pacientes": indices_rango.sum()}
                            for cl in range(n_clases):
                                dict_rango[f"Impacto_Promedio_Clase_{cl}"] = impacto_medio_rango[cl]
                            rangos_resumen.append(dict_rango)
                        else:
                            impacto_medio_rango = np.abs(matriz_shap[indices_rango, X_shap.columns.get_loc(v_num)]).mean()
                            rangos_resumen.append({
                                "Variable": v_num, "Rango": str(rango), 
                                "N_Pacientes": indices_rango.sum(), "Impacto_Promedio": impacto_medio_rango
                            })
                            
        # Exportar CSV de resumen absoluto por rangos
        df_rangos = pd.DataFrame(rangos_resumen)
        df_rangos.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Impacto_Variables_Numericas_{nombre_limpio}.csv"), index=False)
        
        # -------------------------------------------------------------------------
        # ANÁLISIS DIRECCIONAL: VARIABLES NUMÉRICAS
        # -------------------------------------------------------------------------
        # Este análisis evalúa específicamente cómo afecta el rango al riesgo de la clase más alta (+ o -)
        print(f"   -> Calculando impacto direccional ({nombre_efecto_str}) por rangos numéricos...")
        rangos_direccionales = []
        # Aislar tensor enfocado únicamente en la clase crítica (ej. Consumo Alto)
        matriz_clase_alta = matriz_shap[:, :, idx_clase_alta] if n_clases > 1 else matriz_shap
        
        for v_num in vars_num:
            if v_num in X_shap.columns:
                try:
                    bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except:
                    bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                idx_var = X_shap.columns.get_loc(v_num)
                
                for rango in bins_serie.cat.categories:
                    indices_rango = (bins_serie == rango)
                    n_pacientes_rango = indices_rango.sum()
                    
                    if n_pacientes_rango > 0:
                        # Extraer media de impacto crudo para deducir la direccionalidad
                        valores_crudos = matriz_clase_alta[indices_rango, idx_var]
                        promedio_crudo = valores_crudos.mean()
                        
                        if promedio_crudo > 0:
                            efecto = "Aumenta probabilidad (+)"
                        elif promedio_crudo < 0:
                            efecto = "Disminuye probabilidad (-)"
                        else:
                            efecto = "Neutral"

                        # Guardar hallazgos clínicos
                        rangos_direccionales.append({
                            "Variable": v_num,
                            "Rango": str(rango),
                            "N_Pacientes": n_pacientes_rango,
                            f"SHAP_Promedio_Crudo_{target_name}": promedio_crudo,
                            "Efecto_Clinico": efecto
                        })
                        
        # Exportar tabla direccional de variables continuas
        df_direccional = pd.DataFrame(rangos_direccionales)
        df_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccionales_{target_name}_{nombre_limpio}.csv"), index=False)

        # -------------------------------------------------------------------------
        # ANÁLISIS DIRECCIONAL CRUDO CATEGÓRICO (OHE 0 vs 1)
        # -------------------------------------------------------------------------
        # Analiza cómo impacta la presencia o ausencia de una condición al riesgo crítico
        print(f"   -> Calculando impacto direccional ({nombre_efecto_str}) para variables categóricas (OHE)...")
        cat_direccionales = []
        for v_cat in vars_cat_ohe:
            idx_var = X_shap.columns.get_loc(v_cat)
            for valor_cat in [0, 1]:
                indices_cat = (X_shap[v_cat] == valor_cat)
                n_pacientes_cat = indices_cat.sum()
                
                if n_pacientes_cat > 0:
                    promedio_crudo = matriz_clase_alta[indices_cat, idx_var].mean()
                    
                    if promedio_crudo > 0:
                        efecto = "Aumenta probabilidad (+)"
                    elif promedio_crudo < 0:
                        efecto = "Disminuye probabilidad (-)"
                    else:
                        efecto = "Neutral"

                    cat_direccionales.append({
                        "Variable": v_cat,
                        "Condicion_OHE": valor_cat,
                        "Significado": "Presencia (1)" if valor_cat == 1 else "Ausencia (0)",
                        "N_Pacientes": n_pacientes_cat,
                        f"SHAP_Promedio_Crudo_{target_name}": promedio_crudo,
                        "Efecto_Clinico": efecto
                    })
                    
        # Exportar tabla direccional de variables categóricas
        df_cat_direccional = pd.DataFrame(cat_direccionales)
        df_cat_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Categoricas_{target_name}_{nombre_limpio}.csv"), index=False)

        # -------------------------------------------------------------------------
        # PANELES DE DEPENDENCIA MULTICLASE (TOP 10)
        # -------------------------------------------------------------------------
        # Generar sub-gráficos de dispersión para observar relaciones no lineales en cada clase
        print("   -> Generando paneles de dependencia para el Top 10...")
        top_10_vars = df_shap_imp.head(10).index.tolist()
        
        for var in top_10_vars:
            if var in X_shap.columns:
                fig, axes = plt.subplots(1, n_clases, figsize=(5 * n_clases, 4.5))
                # Forzar estructura de array para iteración en caso de que solo haya 1 clase (Binario anómalo)
                if n_clases == 1: axes = [axes]
                
                for clase in range(n_clases):
                    valores_sh_clase = matriz_shap[:, :, clase] if n_clases > 1 else matriz_shap
                    # Renderizar cada sub-gráfico de clase
                    shap.dependence_plot(
                        var, valores_sh_clase, X_shap, 
                        interaction_index=None, ax=axes[clase], show=False
                    )
                    axes[clase].set_title(f'Impacto en clase {clase}', fontsize=10)
                
                # Empaquetar y exportar el panel completo
                fig.suptitle(f'Dependence Plot: {var} ({nombre_limpio})', fontsize=12, y=1.02)
                plt.tight_layout()
                plt.savefig(os.path.join(dir_dependence, f"SHAP_Dependence_{var}.png"), dpi=200, bbox_inches='tight')
                plt.close()
                
        # Limpieza masiva de memoria de todo lo utilizado en este subgrupo para preparar la siguiente iteración
        del X_shap, matriz_shap, df_shap_imp, df_shap_porcentajes, df_direccional, df_cat_direccional; gc.collect()

    # Mensaje de término general
    print("\n" + "="*80)
    print("PROCESO DE ESTRATIFICACIÓN CONCLUIDO CON ÉXITO")
    print(f"Resultados guardados en subcarpetas dentro de:\n{dir_base_estratificado}")
    print("="*80)
    
    # Liberar memoria del dataset madre
    del df_onco_completo; gc.collect()

# Ejemplo de ejecución:
# generar_shap_estratificado_cancer_xgb('SEVERIDAD')
# generar_shap_estratificado_cancer_xgb('CONSUMO_RECURSOS')

In [3]:
# Ejecutar el script para ambos targets
generar_shap_estratificado_cancer_xgb('SEVERIDAD')

INICIANDO SHAP ESTRATIFICADO POR CÁNCER (TOP 10) - TARGET: SEVERIDAD (XGBOOST)
Enfocando análisis direccional en: Severidad Alta (Clase 3)
Hora de inicio: 2026-07-15 04:42:07
-> Cargando modelo XGBoost pre-entrenado desde: Modelo_Optimo_XGBoost_SEVERIDAD.pkl...
-> Inicializando SHAP TreeExplainer nativo...
-> Cargando el 100% de la Cohorte Oncológica de Evaluación...

ADVERTENCIA: La categoría C00_C14 no supera el umbral mínimo de 100 personas. Excluyendo por falta de robustez estadística.

------------------------------------------------------------
--- PROCESANDO SUBGRUPO: C15_C26 (26320 pacientes) ---
------------------------------------------------------------
      -> Procesando bloque 1 de 3...
      -> Procesando bloque 2 de 3...
      -> Procesando bloque 3 de 3...
   -> SHAP completado en 1.53 minutos.
   -> Se excluyeron 17 variables constantes.
   -> Generando CSV de importancias en porcentajes...
   -> Calculando impacto absoluto por rangos numéricos...
   -> Calculando imp

In [2]:
generar_shap_estratificado_cancer_xgb('CONSUMO_RECURSOS')

INICIANDO SHAP ESTRATIFICADO POR CÁNCER (TOP 10) - TARGET: CONSUMO_RECURSOS (XGBOOST)
Enfocando análisis direccional en: Consumo Alto (Clase 2)
Hora de inicio: 2026-07-15 04:02:25
-> Cargando modelo XGBoost pre-entrenado desde: Modelo_Optimo_XGBoost_CONSUMO_RECURSOS.pkl...
-> Inicializando SHAP TreeExplainer nativo...
-> Cargando el 100% de la Cohorte Oncológica de Evaluación...

ADVERTENCIA: La categoría C00_C14 no supera el umbral mínimo de 100 personas. Excluyendo por falta de robustez estadística.

------------------------------------------------------------
--- PROCESANDO SUBGRUPO: C15_C26 (26320 pacientes) ---
------------------------------------------------------------
      -> Procesando bloque 1 de 3...
      -> Procesando bloque 2 de 3...
      -> Procesando bloque 3 de 3...
   -> SHAP completado en 1.31 minutos.
   -> Se excluyeron 17 variables constantes.
   -> Generando CSV de importancias en porcentajes...
   -> Calculando impacto absoluto por rangos numéricos...
   -> Ca